# Neural nets from scratch

**Track:** Training. **Level:** Foundations. **CPU:** yes.

## Goals

- Implement a scalar autograd engine and check a gradient against a finite difference.
- Train a small multilayer perceptron with that engine.
- Train a PyTorch network on the same kind of problem, plot the loss, and apply weight decay.

## Sources and license

Original implementation. The `Value` API follows Andrej Karpathy's micrograd (MIT): addition, multiplication, `tanh`, and a reverse topological walk. The micrograd source file is not copied. The PyTorch loop follows the training pattern in the PyTorch tutorials (BSD-style) and the "from scratch, then a framework" order used by fast.ai (Apache-2.0 course code, not copied) and by Karpathy's Neural Networks: Zero to Hero (MIT).

## Concepts

Backpropagation applies the chain rule on the computation graph. Each node stores the local derivative of its output with respect to its inputs. A reverse walk multiplies those local derivatives by the upstream gradient. A finite difference `(f(x+ε) - f(x-ε)) / (2ε)` is the check that the local derivatives were written down correctly. A training loop is then: forward, loss, backward, step, zero the gradients. Weight decay adds a penalty on the size of the weights. Dropout and weight decay are regularizers. This notebook uses weight decay because its effect on the weight norm is measurable in a few steps.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'numpy', 'torch', 'scikit-learn', 'matplotlib'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

%matplotlib inline
import math
import random

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_moons

from labs._shared.seed import seed_everything

seed_everything(0)
torch.manual_seed(0)


## Guided exercise: reverse-mode autodiff

`Value` records children and a closure that adds the local contribution into `grad`. `backward` walks the graph in an order where every child is finished before its parents are asked to push a gradient. The finite-difference check is on `tanh(x²)` at `x = 1.5`.


In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __pow__(self, exponent):
        out = Value(self.data ** exponent, (self,), f"**{exponent}")

        def _backward():
            self.grad += exponent * (self.data ** (exponent - 1)) * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1.0 - t * t) * out.grad

        out._backward = _backward
        return out

    def backward(self):
        topo, seen = [], set()

        def build(node):
            if node not in seen:
                seen.add(node)
                for child in node._prev:
                    build(child)
                topo.append(node)

        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    def __neg__(self):
        return self * -1

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other if isinstance(other, Value) else -other)

    def __rmul__(self, other):
        return self * other

x = Value(1.5)
y = (x * x).tanh()
y.backward()
eps = 1e-5

def f(v):
    return math.tanh(v * v)

numerical = (f(1.5 + eps) - f(1.5 - eps)) / (2 * eps)
print("analytic", x.grad, "numeric", numerical)
assert abs(x.grad - numerical) < 1e-4


## Guided exercise: a tiny trained network

The targets are linearly separable on purpose. The point of this cell is the training loop on the scalar engine: forward, squared error, `backward`, a gradient step, and an explicit zeroing of every `grad`. The PyTorch section is where the model has to fit a curve.


In [ ]:
class Neuron:
    def __init__(self, n_in):
        self.w = [Value(random.uniform(-1.0, 1.0)) for _ in range(n_in)]
        self.b = Value(0.0)

    def __call__(self, x):
        act = self.b
        for wi, xi in zip(self.w, x):
            act = act + wi * xi
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, n_in, n_out):
        self.neurons = [Neuron(n_in) for _ in range(n_out)]

    def __call__(self, x):
        out = [neuron(x) for neuron in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]


class MLP:
    def __init__(self, sizes):
        self.layers = [Layer(a, b) for a, b in zip(sizes, sizes[1:])]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


points = [((random.uniform(-2, 2), random.uniform(-2, 2))) for _ in range(24)]
targets = [1.0 if p[0] > 0 else -1.0 for p in points]
net = MLP([2, 8, 1])
losses = []
for step in range(40):
    total = Value(0.0)
    correct = 0
    for (x0, x1), target in zip(points, targets):
        pred = net([Value(x0), Value(x1)])
        total = total + (pred - target) ** 2
        correct += int((pred.data > 0) == (target > 0))
    loss = total * (1.0 / len(points))
    for p in net.parameters():
        p.grad = 0.0
    loss.backward()
    for p in net.parameters():
        p.data += -0.2 * p.grad
    losses.append(loss.data)
    if step in (0, 39):
        print(f"step {step} loss {loss.data:.3f} acc {correct / len(points):.2f}")
assert losses[-1] < losses[0]


## Guided exercise: PyTorch training loop

Two-moons is not linearly separable. The network is a two-hidden-layer MLP. One run uses AdamW with `weight_decay=0`. The other uses the same learning rate and `weight_decay=0.05`. Both should drive the training loss down. The decayed run should finish with a smaller parameter norm. Validation accuracy is printed. It is not the thing the assertion treats as a leaderboard.


In [ ]:
class TorchMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32), nn.ReLU(),
            nn.Linear(32, 32), nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)


def run(weight_decay: float, steps: int = 120):
    torch.manual_seed(0)
    Xm, ym = make_moons(n_samples=240, noise=0.18, random_state=0)
    X = torch.tensor(Xm, dtype=torch.float32)
    y = torch.tensor(ym, dtype=torch.float32).view(-1, 1)
    n = int(0.75 * len(X))
    model = TorchMLP()
    opt = torch.optim.AdamW(model.parameters(), lr=0.04, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()
    train_losses, val_losses = [], []
    for _ in range(steps):
        model.train()
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(model(X[:n]), y[:n])
        loss.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            train_losses.append(float(loss_fn(model(X[:n]), y[:n])))
            val_losses.append(float(loss_fn(model(X[n:]), y[n:])))
    with torch.no_grad():
        acc = float(((model(X[n:]) > 0).float() == y[n:]).float().mean())
        norm = float(torch.sqrt(sum(p.detach().pow(2).sum() for p in model.parameters())))
    return train_losses, val_losses, acc, norm

free_train, free_val, free_acc, free_norm = run(0.0)
decay_train, decay_val, decay_acc, decay_norm = run(0.05)
print(f"no decay  val acc {free_acc:.2f}  weight norm {free_norm:.2f}  final train loss {free_train[-1]:.3f}")
print(f"decay     val acc {decay_acc:.2f}  weight norm {decay_norm:.2f}  final train loss {decay_train[-1]:.3f}")
assert free_train[-1] < free_train[0]
assert decay_train[-1] < decay_train[0]
assert decay_norm < free_norm
assert free_acc > 0.8 and decay_acc > 0.8

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(free_train, label="train, no decay")
ax.plot(free_val, label="val, no decay")
ax.plot(decay_val, label="val, decay 0.05")
ax.set_xlabel("step")
ax.set_ylabel("BCE loss")
ax.legend()
ax.set_title("Two moons")
plt.tight_layout()
plt.show()
print("LAB OK")


## Your turn

The finite-difference check used `x = 1.5`. Repeat it at `x = -0.3` and confirm the analytic gradient still matches.

## Solution


In [ ]:
z = Value(-0.3)
out = (z * z).tanh()
out.backward()
num = (f(-0.3 + eps) - f(-0.3 - eps)) / (2 * eps)
print(z.grad, num)
assert abs(z.grad - num) < 1e-4


## Beyond the course

Train the moons network with SGD at learning rate 0.1 and at learning rate 5, from the same initialization recipe (the seed is reset inside `run_sgd`). The large step size is the ablation. Report the final training loss of each. The stable run should finish lower.


In [ ]:
def run_sgd(lr: float, steps: int = 60) -> float:
    torch.manual_seed(0)
    Xm, ym = make_moons(n_samples=200, noise=0.15, random_state=0)
    X = torch.tensor(Xm, dtype=torch.float32)
    y = torch.tensor(ym, dtype=torch.float32).view(-1, 1)
    model = TorchMLP()
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    last = float("nan")
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(model(X), y)
        loss.backward()
        opt.step()
        last = float(loss.detach())
    return last

low, high = run_sgd(0.1), run_sgd(5.0)
print("lr 0.1", round(low, 3), "lr 5", round(high, 3))
assert low < high


## Research questions

1. The scalar engine stores every intermediate `Value`. What does PyTorch store instead, and what does that change about memory during the backward pass?
2. Weight decay lowered the parameter norm. Did validation loss move in the same direction on this dataset? If not, what would you need to see before calling the decay run "better"?
3. `tanh` saturates. Where in the finite-difference check would a gradient near zero be correct rather than a bug?

## Further reading

- Karpathy, micrograd (MIT) and *Neural Networks: Zero to Hero*. The engine above is an original implementation of that design. https://github.com/karpathy/micrograd
- PyTorch tutorials, "Learning PyTorch with Examples" (BSD-style).
- fast.ai Practical Deep Learning, the from-scratch lesson (Apache-2.0 course code). Not copied.
- Goodfellow, Bengio, and Courville, *Deep Learning*, chapter 6, on the chain rule.
